# Aim
Use the established SignalStreamer to stream 1090 data and continuously store aircraft datacoverage as possible

Assess a range of different parameterisations with the intent of trying to have as close to continuous 


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import planesailing as ps

from planesailing import main

import time
import pandas as pd
import numpy as np
import scipy
from copy import copy

import cosmosdr
import cosmosdr.signal_acquisition as s_acq
import cosmosdr.signal_processing as s_proc

import structlog
logger = structlog.get_logger()

try:
    sdr.close()
except:
    pass

# import plotly
# from plotly.graph_objects import Scatter
# from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
# init_notebook_mode(connected=False)
# import plotly.express as px
# import plotly.graph_objects as go

pd.options.display.max_columns = 99

In [ ]:
ADSB_FREQUENCY = 1090e6
# Total number of bits in a single ADSB message
ADSB_BITS = 112
# One microsecond timeslot for one bit, 'on' if signal is within the first half of this slot
ADSB_SLOT_LENGTH = 1 / 1e6

center_freq = ADSB_FREQUENCY
# reccomended upper limit of sample rate. Fast enough to oversample
sample_rate = 2.4e6


# choose integer samples per microsecond: 12 samples/us
# This is a good choice, because 6x2 = 12, and 5*2.4=12, so we can sample to 12, then subsample back to 1 block per 0.5us
target_sr = 12e6
samples_per_us_after_upsampling = int(round(target_sr/1e6))  # 12

# Grid search for optimal parameters to maximise parsed ADSB messages

In [ ]:
results = []
n_samples_per_read = 256
for sleep_time in [0.01, 0.025, 0.05]:
    for n_reads_per_acquisition in [16, 32, 64]:
        for n_samples_per_read in [2048, 4096, 8192]:
            res, delta_t = ps.main.stream_and_decode_adsb(
                n=100,
                sleep_time=sleep_time,
                n_reads_per_acquisition=n_reads_per_acquisition,
                n_samples_per_read=n_samples_per_read,
            )
    
            results.append({
                "sleep_time":sleep_time,
                "n_reads_per_acquisition":n_reads_per_acquisition,
                "n_samples_per_read":n_samples_per_read,
                "res_n":len(res),
                "delta_t":delta_t
            })
results = pd.DataFrame(results)

In [ ]:
results.sort_values("res_n", ascending=False)

In [ ]:
results = []
n_samples_per_read = 256
for sleep_time in [0.0025, 0.005, 0.01]:
    for n_reads_per_acquisition in [32, 64]:
        for n_samples_per_read in [1024, 2048, 4096]:
            res, delta_t = ps.main.stream_and_decode_adsb(
                n=100,
                sleep_time=sleep_time,
                n_reads_per_acquisition=n_reads_per_acquisition,
                n_samples_per_read=n_samples_per_read,
            )
    
            results.append({
                "sleep_time":sleep_time,
                "n_reads_per_acquisition":n_reads_per_acquisition,
                "n_samples_per_read":n_samples_per_read,
                "res_n":len(res),
                "delta_t":delta_t
            })
results = pd.DataFrame(results)

In [ ]:
results

# Same test, with increased indices 5 -> 20

In [ ]:
results = []
n_samples_per_read = 256
for sleep_time in [0.0025, 0.005, 0.01]:
    for n_reads_per_acquisition in [32, 64]:
        for n_samples_per_read in [1024, 2048, 4096]:
            res, delta_t = ps.main.stream_and_decode_adsb(
                n=100,
                sleep_time=sleep_time,
                n_reads_per_acquisition=n_reads_per_acquisition,
                n_samples_per_read=n_samples_per_read,
            )
    
            results.append({
                "sleep_time":sleep_time,
                "n_reads_per_acquisition":n_reads_per_acquisition,
                "n_samples_per_read":n_samples_per_read,
                "res_n":len(res),
                "delta_t":delta_t
            })
results = pd.DataFrame(results)

In [ ]:
# Interesting!! The best params were the same, but the hits increased substantially
# This implies that I'm simply missing a bunch of reads 
# I blame the approach of 'take the strongest signal' - this seems to bias towards big passenger jets
# Clearly, trawling through the lower strength signals is finding more that can be used
results.sort_values("res_n", ascending=False)

# Push further in the direction of success

In [ ]:
results_big = []
n_samples_per_read = 256
for sleep_time in [0.005, 0.01, 0.02]:
    for n_reads_per_acquisition in [64, 128, 256]:
        for n_samples_per_read in [2048, 4096, 4096*2]:
            res, delta_t = ps.main.stream_and_decode_adsb(
                n=100,
                sleep_time=sleep_time,
                n_reads_per_acquisition=n_reads_per_acquisition,
                n_samples_per_read=n_samples_per_read,
            )
    
            results_big.append({
                "sleep_time":sleep_time,
                "n_reads_per_acquisition":n_reads_per_acquisition,
                "n_samples_per_read":n_samples_per_read,
                "res_n":len(res),
                "delta_t":delta_t
            })
results_big = pd.DataFrame(results_big)

In [ ]:
results.sort_values("res_n", ascending=False)

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=20, verbose=True
)

In [ ]:
delta_t

In [ ]:
res

In [ ]:
import plotly.express as px
import pandas as pd

# import cufflinks as cf
# cf.go_offline()
# 1. Sample Data (Replace with your actual data)

# 2. Create the scatter geo plot
fig = px.scatter_map(res,
                     lat='latitude',
                     lon='longitude',
                     hover_name='icao24', # Show city name on hover
                     size='altitude', # Size points by population
                     color='icao24', # Color points by population
                     title='Planes!',
                    width=1600, height=800)

# 3. Display the plot
fig.show()

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=100, n_samples_per_read=2048*3 # increase it, now we're actually using more (looking at the top 5, not just the top)
)

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=100, n_samples_per_read=2048*3 # increase it, now we're actually using more (looking at the top 5, not just the top 1)
)

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=500, n_samples_per_read=2048*3 # increase it, now we're actually using more (looking at the top 5, not just the top 1)
)

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=500, n_samples_per_read=2048*2 # increase it, now we're actually using more (looking at the top 5, not just the top 1)
)

In [ ]:
res, delta_t = ps.main.stream_and_decode_adsb(
    n=500, n_samples_per_read=2048*4 # increase it, now we're actually using more (looking at the top 5, not just the top 1)
)

In [ ]:
res

In [ ]:
res